# 12 · Random Forest + Kriging

Template para `RegressionKrigingModel`, separando claramente la parte no lineal del bosque y el ajuste espacial sobre residuos.

## Hipótesis del modelo

- El Random Forest captura relaciones no lineales y umbrales.
- El kriging sobre residuos agrega estructura espacial que el bosque no explica por sí solo.
- La interpretación puede combinar importancia global de features con mapas de residuo antes y después del kriging.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ml_core.models.rfrkModel import RegressionKrigingModel

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "12_rf_kriging"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Datos y configuración

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rental_listings_model_input.parquet"
# df = pd.read_parquet(DATA_PATH)

target_col = "price_m2"
coord_cols = ["lon", "lat"]
feature_cols = []

rf_params = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
}

kriging_params = {
    "n_closest_points": 10,
    "variogram_model": "linear",
    "pseudo_inv": True,
    "pseudo_inv_type": "pinvh",
}

## Entrenamiento

In [ ]:
# X_train = train_df[feature_cols]
# y_train = train_df[target_col]
# coords_train = train_df[coord_cols].to_numpy()
# model = RegressionKrigingModel(rf_params=rf_params, kriging_params=kriging_params)
# model.fit(X_train, y_train, coords_train)
# model

## Tuning tentativo

Acá podés barrer hiperparámetros del bosque y del kriging, o hacer primero tuning del RF y después del componente espacial.

In [ ]:
# rf_param_grid = {
#     "n_estimators": [200, 400],
#     "max_depth": [None, 12, 20],
#     "min_samples_leaf": [1, 3],
# }
# model.tune_hyperparameters(X_train, y_train.to_numpy().reshape(-1, 1), coords_train, rf_param_grid=rf_param_grid)
# tuned_config = {"rf_params": model.rf_params, "kriging_params": model.kriging_params}
# tuned_config

## Evaluación global

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residuals = y_true - y_pred
    return {
        "rmse": float(np.sqrt(np.mean(residuals ** 2))),
        "mae": float(np.mean(np.abs(residuals))),
        "r2": float(1 - np.sum(residuals ** 2) / np.sum((y_true - y_true.mean()) ** 2)),
        "bias": float(residuals.mean()),
    }

# X_test = test_df[feature_cols]
# coords_test = test_df[coord_cols].to_numpy()
# y_test = test_df[target_col]
# y_pred = model.predict(X_test, coords_test)
# metrics = regression_metrics(y_test, y_pred)
# metrics

## Interpretación del Random Forest

In [ ]:
# feature_importance = pd.DataFrame({
#     "feature": feature_cols,
#     "importance": model.feature_importances_(),
# }).sort_values("importance", ascending=False)
# feature_importance

## Residuo del bosque vs residuo final

Si querés entender el aporte del kriging, compará explícitamente el error del RF puro contra el error final del modelo combinado.

In [ ]:
# rf_only_pred = model.rf_.predict(X_test)
# comparison = pd.DataFrame({
#     "y_true": y_test,
#     "rf_pred": np.asarray(rf_only_pred).reshape(-1),
#     "rk_pred": np.asarray(y_pred).reshape(-1),
# })
# comparison["rf_residual"] = comparison["y_true"] - comparison["rf_pred"]
# comparison["rk_residual"] = comparison["y_true"] - comparison["rk_pred"]
# comparison.head()

## Export

In [ ]:
# test_export = test_df[[target_col] + coord_cols].copy()
# test_export = test_export.rename(columns={target_col: "y_true"})
# test_export["y_pred"] = np.asarray(y_pred).reshape(-1)
# test_export["residual"] = test_export["y_true"] - test_export["y_pred"]
# test_export["rf_only_pred"] = np.asarray(rf_only_pred).reshape(-1)
# test_export["split"] = "test"
# test_export.to_parquet(OUTPUT_DIR / "test_predictions.parquet", index=False)
# feature_importance.to_parquet(OUTPUT_DIR / "interpretability.parquet", index=False)
# run_config = {"rf_params": rf_params, "kriging_params": kriging_params}
# (OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
# (OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))